# Random Effects & Mixed Models

A fixed-effect analysis compares the particular levels under study. A random-effect analysis treats observed levels as draws from a population and asks how much variation that population contributes. The distinction comes from the sampling plan and inferential goal, not whether the factor is numeric or categorical.

## One Random Factor

For $a$ groups and $n$ observations per group,

$$y_{ij}=\mu+u_i+\epsilon_{ij},\qquad
u_i\sim N(0,\sigma_u^2),\quad\epsilon_{ij}\sim N(0,\sigma^2),$$

where $u_i$ are independent group effects and all residuals are independent of them. The mean of the group-effect distribution is zero; the realized group effects need not sum to zero.

The marginal variance is $\sigma_u^2+\sigma^2$. Two distinct observations in the same group share $u_i$, so their covariance is $\sigma_u^2$. Consequently,

$$\rho=\frac{\sigma_u^2}{\sigma_u^2+\sigma^2}$$

is the **intraclass correlation**. It measures within-group similarity and the between-group fraction of marginal variance in this model.

## Expected Mean Squares

The balanced ANOVA arithmetic is unchanged from a one-way fixed model. Its interpretation changes:

| Source | df | Expected mean square |
|---|---:|---|
| Groups | $a-1$ | $\sigma^2+n\sigma_u^2$ |
| Error | $a(n-1)$ | $\sigma^2$ |

Thus the normal-model test of $H_0:\sigma_u^2=0$ uses $F=MS_A/MS_E$ with $(a-1,a(n-1))$ df. The alternative is positive variance, so use the upper tail.

Method-of-moments estimates are

$$\widehat\sigma^2=MS_E,\qquad \widehat\sigma_u^2=\frac{MS_A-MS_E}{n}.$$

:::{dropdown} Why the group mean square contains a factor of n
A group mean has variance $\operatorname{Var}(\bar y_{i\cdot})=\sigma_u^2+\sigma^2/n$. The between-group mean square is $n$ times the sample variance of the independent group means. Its expectation is therefore $n\sigma_u^2+\sigma^2$.
:::

For an illustrative experiment with $a=5$, $n=4$, $MS_A=18$ and $MS_E=2$, the estimates are $\widehat\sigma_u^2=4$, $\widehat\sigma^2=2$, and $\widehat\rho=2/3$.

In [1]:
from scipy.stats import f
MSA, MSE, a, n = 18., 2., 5, 4
between = (MSA-MSE)/n
print('Group variance:', between)
print('Residual variance:', MSE)
print('ICC:', round(between/(between+MSE), 4))
print('F:', MSA/MSE, 'p:', round(f.sf(MSA/MSE, a-1, a*(n-1)), 6))

Group variance: 4.0
Residual variance: 2.0
ICC: 0.6667
F: 9.0 p: 0.00065


A negative moment estimate can occur when $MS_A<MS_E$. It does not imply a negative population variance. Report the estimation issue and use an appropriate nonnegative estimation method, such as constrained REML, when fitting the model; simply truncating at zero changes the estimator.

## Two Crossed Random Factors

For $a$ levels of A, $b$ levels of B, and $n$ replicates per cell,

$$y_{ijk}=\mu+u_i+v_j+w_{ij}+\epsilon_{ijk}.$$

Assume mutually independent zero-mean normal effects with variances $\sigma_A^2,\sigma_B^2,\sigma_{AB}^2,\sigma^2$ respectively.

| Source | df | Expected mean square | Denominator for testing its variance component |
|---|---:|---|---|
| A | $a-1$ | $\sigma^2+n\sigma_{AB}^2+bn\sigma_A^2$ | $MS_{AB}$ |
| B | $b-1$ | $\sigma^2+n\sigma_{AB}^2+an\sigma_B^2$ | $MS_{AB}$ |
| AB | $(a-1)(b-1)$ | $\sigma^2+n\sigma_{AB}^2$ | $MS_E$ |
| Error | $ab(n-1)$ | $\sigma^2$ | — |

For example, under $H_0:\sigma_A^2=0$, $MS_A$ and $MS_{AB}$ estimate the same quantity. Comparing A with $MS_E$ would leave interaction variation in the numerator only. Normality and the balanced orthogonal design supply the independent scaled chi-square distributions needed for these exact F tests; matching expectations alone is not sufficient in arbitrary models.

Subtracting expected mean squares gives

$$\widehat\sigma_{AB}^2=\frac{MS_{AB}-MS_E}{n},\quad
\widehat\sigma_A^2=\frac{MS_A-MS_{AB}}{bn},\quad
\widehat\sigma_B^2=\frac{MS_B-MS_{AB}}{an}.$$

### Covariance Depends on What Is Shared

Let $V=\sigma_A^2+\sigma_B^2+\sigma_{AB}^2+\sigma^2$. Distinct observations sharing only A have covariance $\sigma_A^2$; sharing only B gives $\sigma_B^2$; sharing the same cell gives $\sigma_A^2+\sigma_B^2+\sigma_{AB}^2$. Divide the appropriate covariance by $V$ for the correlation. A single interaction variance fraction is not, by itself, the same-cell correlation.

## A Fixed, B Random: Restricted Mixed Model

Replace $u_i$ by fixed constants $\tau_i$, with $\sum_i\tau_i=0$. In the **restricted** mixed model, require

$$\sum_{i=1}^a w_{ij}=0\quad\text{for every }j.$$

One construction is $w_{ij}=q_{ij}-\bar q_{\cdot j}$ for independent $q_{ij}\sim N(0,\sigma_{AB}^2)$. This yields

$$\operatorname{Var}(w_{ij})=\frac{a-1}{a}\sigma_{AB}^2,\qquad
\operatorname{Cov}(w_{ij},w_{i'j})=-\frac1a\sigma_{AB}^2\quad(i\ne i').$$

Here $\sigma_{AB}^2$ is the covariance-scale parameter; it is not the variance of a single constrained interaction. Different B groups are independent, and $v_j$ and residuals are independent of the interactions.

:::{dropdown} Derive the negative covariance
Since $\operatorname{Var}(\bar q_{\cdot j})=\sigma_{AB}^2/a$ and $\operatorname{Cov}(q_{ij},\bar q_{\cdot j})=\sigma_{AB}^2/a$, subtracting the group mean gives variance $\sigma_{AB}^2(1-1/a)$. For distinct $i,i'$, the covariance is $0-\sigma_{AB}^2/a-\sigma_{AB}^2/a+\sigma_{AB}^2/a=-\sigma_{AB}^2/a$.
:::

The expected mean squares become

$$E[MS_A]=\sigma^2+n\sigma_{AB}^2+\frac{bn}{a-1}\sum_i\tau_i^2,$$

$$E[MS_B]=\sigma^2+an\sigma_B^2,\qquad
E[MS_{AB}]=\sigma^2+n\sigma_{AB}^2,\qquad E[MS_E]=\sigma^2.$$

| Null hypothesis | F statistic | Numerator / denominator df |
|---|---|---|
| All $\tau_i=0$ | $MS_A/MS_{AB}$ | $a-1$ / $(a-1)(b-1)$ |
| $\sigma_B^2=0$ | $MS_B/MS_E$ | $b-1$ / $ab(n-1)$ |
| $\sigma_{AB}^2=0$ | $MS_{AB}/MS_E$ | $(a-1)(b-1)$ / $ab(n-1)$ |

The interaction cancels when averaging across all fixed A levels within a B level. That is why it is absent from $E[MS_B]$. It does not cancel from contrasts between A levels, which explains the interaction denominator for testing A.

For this restricted model, $\widehat\sigma_B^2=(MS_B-MS_E)/(an)$. There is no $\sigma_A^2$ to estimate because A has fixed effects.

### Do Not Mix Model Conventions

An **unrestricted** mixed model uses independent $w_{ij}$ without the sum-to-zero restriction. Then $E[MS_B]$ includes $n\sigma_{AB}^2$, and the B test uses $MS_{AB}$. The [comparison of mixed-model conventions](https://online.stat.psu.edu/stat502/Lesson06) explains this distinction. Identify the assumptions used by a software fit before choosing an ANOVA denominator.

## Practical Limits

These simple tables assume a balanced design, common residual variance, independent random components, and sufficient replication. With $n=1$, interaction and residual variance cannot be separated by the usual cell replication. Missing cells or unequal group sizes generally require a model-based mixed-effects analysis rather than these plug-in formulas.

For a one-way model, $\operatorname{Var}(\bar y_{\cdot\cdot})=\sigma_u^2/a+\sigma^2/(an)$. More observations per group reduce residual uncertainty, but adding independent groups is needed to reduce the group component effectively. Treating all observations as independent ignores this distinction.